## MolReps

In [14]:
from pathlib import Path
import pandas as pd

tmp_dir = Path("results")
conversion_dir = tmp_dir / "mol_rep_conversion" / "v1.1"
ocr_dir = tmp_dir / "mol_rep_ocr" / "v1.1"
rollout_dir = tmp_dir / "pass_at_k"

In [43]:
from src.utils import compute_mol_metrics

def aggregate_func(df):
    return pd.Series({
        "num_samples": df.shape[0],
        "gt_valid_ratio": df["is_gt_valid"].mean() * 100,
        "pred_valid_ratio": df["is_pred_valid"].mean() * 100,
        "em_ratio": df["is_em"].mean() * 100,
        "can_smiles_match_ratio": df["is_can_smiles_match"].mean() * 100,
        "inchikey_match_ratio": df["is_inchikey_match"].mean() * 100,
        "tanimoto_sim_mean": df["tanimoto_sim"].mean(),
        "tanimoto_sim_median": df["tanimoto_sim"].median(),
        "tanimoto_sim_q1": df["tanimoto_sim"].quantile(0.25),
        "tanimoto_sim_q3": df["tanimoto_sim"].quantile(0.75),
        "tanimoto_sim_std": df["tanimoto_sim"].std(),
    })

In [16]:
conversion_raw_reponses = pd.DataFrame()

for file in (conversion_dir / "raw_responses").glob("*.jsonl"):
    df = pd.read_json(file, lines=True)
    conversion_raw_reponses = pd.concat([conversion_raw_reponses, df], axis=0, ignore_index=True)

raw_conversion_df = conversion_raw_reponses.copy()

raw_conversion_df[['is_gt_valid', 'is_pred_valid', 'is_em', 'is_can_smiles_match', 'is_inchikey_match', 'tanimoto_sim', '_parse_status']] = conversion_raw_reponses.apply(lambda row: compute_mol_metrics(row["completion"], row["raw_responses"], row["output_rep_type"]), axis=1, result_type="expand")


extra_conversion_raw_responses = pd.DataFrame()

for file in (conversion_dir / "extra_test" / "raw_responses").glob("*.jsonl"):
    df = pd.read_json(file, lines=True)
    extra_conversion_raw_responses = pd.concat([extra_conversion_raw_responses, df], axis=0, ignore_index=True)

raw_extra_conversion_df = extra_conversion_raw_responses.copy()

raw_extra_conversion_df[['is_gt_valid', 'is_pred_valid', 'is_em', 'is_can_smiles_match', 'is_inchikey_match', 'tanimoto_sim', '_parse_status']] = extra_conversion_raw_responses.apply(lambda row: compute_mol_metrics(row["completion"], row["raw_responses"], row["output_rep_type"]), axis=1, result_type="expand")

In [17]:
ocr_raw_responses = pd.DataFrame()

for file in (ocr_dir / "raw_responses").glob("*.jsonl"):
    df = pd.read_json(file, lines=True)
    ocr_raw_responses = pd.concat([ocr_raw_responses, df], axis=0, ignore_index=True)

raw_ocr_df = ocr_raw_responses.copy()

raw_ocr_df[['is_gt_valid', 'is_pred_valid', 'is_em', 'is_can_smiles_match', 'is_inchikey_match', 'tanimoto_sim', '_parse_status']] = ocr_raw_responses.apply(lambda row: compute_mol_metrics(row["completion"], row["raw_responses"], row["output_rep_type"]), axis=1, result_type="expand")


extra_ocr_raw_responses = pd.DataFrame()

for file in (ocr_dir / "extra_test" / "raw_responses").glob("*.jsonl"):
    df = pd.read_json(file, lines=True)
    extra_ocr_raw_responses = pd.concat([extra_ocr_raw_responses, df], axis=0, ignore_index=True)

raw_extra_ocr_df = extra_ocr_raw_responses.copy()

raw_extra_ocr_df[['is_gt_valid', 'is_pred_valid', 'is_em', 'is_can_smiles_match', 'is_inchikey_match', 'tanimoto_sim', '_parse_status']] = extra_ocr_raw_responses.apply(lambda row: compute_mol_metrics(row["completion"], row["raw_responses"], row["output_rep_type"]), axis=1, result_type="expand")

In [18]:
conversion_df = raw_conversion_df.copy()
extra_conversion_df = raw_extra_conversion_df.copy()

In [19]:
ocr_df = raw_ocr_df.copy()
extra_ocr_df = raw_extra_ocr_df.copy()

In [20]:
col_names = {
    "model_name": "Model",
    "model_stage": "Model Version",
    "task_type": "Tasks",
    "input_rep_type": "Input Rep. Type",
    "output_rep_type": "Output Rep. Type",
}

conversion_df = conversion_df.rename(columns=col_names)
extra_conversion_df = extra_conversion_df.rename(columns=col_names)
ocr_df = ocr_df.rename(columns=col_names)
extra_ocr_df = extra_ocr_df.rename(columns=col_names)

In [21]:
def _rename_model_name(df):
    tmp_map = {
        "qwen3_4b_i": "Q3-4B-I",
        "qwen3_vl_4b_i": "Q3-VL-4B-I",
        "qwen3_4b_i_sft_lora_conversion": "MQ3-4B-I",
        "qwen3_vl_4b_i_sft_lora_ocr_conversion": "MQ3-VL-4B-I",
        "qwen3_vl_4b_i_sft_lora_ocr": "MQ3-VL-4B-I\\_{{OCR}}",
        "qwen3_vl_4b_i_sft_lora_conversion": "MQ3-VL-4B-I\\_{{Conv.}}",
    }
    df["Model"] = df["Model"].replace(tmp_map)
    model_name_order = [
        "Q3-4B-I",
        "Q3-VL-4B-I",
        "MQ3-4B-I",
        "MQ3-VL-4B-I\\_{{OCR}}",
        "MQ3-VL-4B-I\\_{{Conv.}}",
        "MQ3-VL-4B-I",
    ]
    df["Model"] = pd.Categorical(df["Model"], categories=model_name_order, ordered=True)
    return df

conversion_df = _rename_model_name(conversion_df)
extra_conversion_df = _rename_model_name(extra_conversion_df)
ocr_df = _rename_model_name(ocr_df)
extra_ocr_df = _rename_model_name(extra_ocr_df)


def _rename_task_type(df):
    task_names = {
        "can_reps_translation": "Rep. Trans.",
        "iupac_understanding": "IUPAC Und.",
        "cml_understanding": "CML Und.",
        "intra_rep_normalization": "Intra Norm.",
        "cross_rep_normalization": "Cross Norm.",
        "ocr": "OCR",
    }
    df["Tasks"] = df["Tasks"].replace(task_names)
    task_type_order = [
        "Rep. Trans.",
        "IUPAC Und.",
        "CML Und.",
        "Intra Norm.",
        "Cross Norm.",
        "OCR"
    ]
    df["Tasks"] = pd.Categorical(df["Tasks"], categories=task_type_order, ordered=True)
    return df

conversion_df = _rename_task_type(conversion_df)
extra_conversion_df = _rename_task_type(extra_conversion_df)
ocr_df = _rename_task_type(ocr_df)
extra_ocr_df = _rename_task_type(extra_ocr_df)


def _rename_output_rep_type(df):
    output_rep_type_names = {
        "can_smiles": "SMILES",
        "can_selfies": "SELFIES",
        "can_deepsmiles": "DeepSMILES",
        "inchi": "InChI",
    }
    df["Output Rep. Type"] = df["Output Rep. Type"].replace(output_rep_type_names)
    output_rep_type_order = [
        "SMILES",
        "SELFIES",
        "DeepSMILES",
        "InChI"
    ]
    df["Output Rep. Type"] = pd.Categorical(df["Output Rep. Type"], categories=output_rep_type_order, ordered=True)
    return df

conversion_df = _rename_output_rep_type(conversion_df)
extra_conversion_df = _rename_output_rep_type(extra_conversion_df)
ocr_df = _rename_output_rep_type(ocr_df)
extra_ocr_df = _rename_output_rep_type(extra_ocr_df)


def _rename_model_version(df):
    model_version_names = {
        "base": "Base",
        "sft": "Trained"
    }
    df["Model Version"] = df["Model Version"].replace(model_version_names)
    model_version_order = [
        "Base",
        "Trained",
    ]
    df["Model Version"] = pd.Categorical(df["Model Version"], categories=model_version_order, ordered=True)
    return df

conversion_df = _rename_model_version(conversion_df)
extra_conversion_df = _rename_model_version(extra_conversion_df)
ocr_df = _rename_model_version(ocr_df)
extra_ocr_df = _rename_model_version(extra_ocr_df)


def _rename_input_rep_type(df):
    input_rep_type_names = {
        "can_smiles": "SMILES",
        "can_selfies": "SELFIES",
        "can_deepsmiles": "DeepSMILES",
        "random_smiles": "R. SMILES",
        "random_selfies": "R. SELFIES",
        "random_deepsmiles": "R. DeepSMILES",
        "inchi": "InChI",
        "iupac": "IUPAC",
        "cml": "CML",
        "mol_image": "Mol. Image",
    }
    df["Input Rep. Type"] = df["Input Rep. Type"].replace(input_rep_type_names)
    input_rep_type_order = [
        "SMILES",
        "SELFIES",
        "DeepSMILES",
        "InChI",
        "R. SMILES",
        "R. SELFIES",
        "R. DeepSMILES",
        "IUPAC",
        "CML",
        "Mol. Image"
    ]
    df["Input Rep. Type"] = pd.Categorical(df["Input Rep. Type"], categories=input_rep_type_order, ordered=True)
    return df

conversion_df = _rename_input_rep_type(conversion_df)
extra_conversion_df = _rename_input_rep_type(extra_conversion_df)
ocr_df = _rename_input_rep_type(ocr_df)
extra_ocr_df = _rename_input_rep_type(extra_ocr_df)

In [22]:
agg_col_names = {
    "num_samples": "N",
    "gt_valid_ratio": "GT Valid. (\\%)",
    "pred_valid_ratio": "Valid. (\\%)",
    "em_ratio": "EM (\\%)",
    "can_smiles_match_ratio": "CSM (\\%)",
    "inchikey_match_ratio": "IKM (\\%)",
    "tanimoto_sim_mean": "T. Sim",
    "tanimoto_sim_std": "T. Sim (std)"
}

### Table1: Base v.s. MolQwen3

In [23]:
tmp_df = conversion_df.groupby(["Model", "Tasks"]).apply(aggregate_func).reset_index()
tmp_df = tmp_df.rename(columns=agg_col_names)

# 计算 Overall（按 Model 分组，忽略 Tasks）
overall_df = conversion_df.groupby("Model").apply(aggregate_func).reset_index()
overall_df = overall_df.rename(columns=agg_col_names)
overall_df["Tasks"] = "Overall"

# 设置 Overall 的 Categorical 顺序（放在最后）
all_tasks = ["Rep. Trans.", "IUPAC Und.", "CML Und.", "Intra Norm.", "Cross Norm.", "Overall"]
overall_df["Tasks"] = pd.Categorical(overall_df["Tasks"], categories=all_tasks, ordered=True)
tmp_df["Tasks"] = pd.Categorical(tmp_df["Tasks"], categories=all_tasks, ordered=True)

# 合并
tmp_df = pd.concat([tmp_df, overall_df], ignore_index=True)
tmp_df = tmp_df.sort_values(["Model", "Tasks"])

tmp_df = tmp_df.round(3)
tmp_df

/tmp/ipykernel_27792/1830000253.py:12: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  tmp_df["Tasks"] = pd.Categorical(tmp_df["Tasks"], categories=all_tasks, ordered=True)


,Model,Tasks,N,GT Valid. (\%),Valid. (\%),EM (\%),CSM (\%),IKM (\%),T. Sim,T. Sim (std)
0,Q3-4B-I,Rep. Trans.,247.0,100.0,14.980,0.000,0.000,0.000,0.007,0.027
1,Q3-4B-I,IUPAC Und.,69.0,100.0,27.536,0.000,0.000,0.000,0.025,0.074
2,Q3-4B-I,CML Und.,76.0,100.0,11.842,0.000,0.000,0.000,0.004,0.015
3,Q3-4B-I,Intra Norm.,114.0,100.0,49.123,0.000,30.702,30.702,0.385,0.455
4,Q3-4B-I,Cross Norm.,369.0,100.0,15.176,0.000,0.000,0.000,0.014,0.077
30,Q3-4B-I,Overall,875.0,100.0,20.229,0.000,4.000,4.000,0.060,0.214
5,Q3-VL-4B-I,Rep. Trans.,247.0,100.0,5.668,0.405,0.405,0.405,0.011,0.076
6,Q3-VL-4B-I,IUPAC Und.,69.0,100.0,13.043,0.000,0.000,0.000,0.020,0.071
7,Q3-VL-4B-I,CML Und.,76.0,100.0,17.105,0.000,0.000,0.000,0.010,0.030
8,Q3-VL-4B-I,Intra Norm.,114.0,100.0,70.175,0.000,53.509,53.509,0.617,0.463


In [24]:
clean_df = tmp_df.copy()
clean_df = clean_df.drop(columns=["N", "GT Valid. (\\%)"])
clean_df = clean_df.drop(columns=["T. Sim (std)"])
clean_df = clean_df[~clean_df.Model.isin(["MQ3-VL-4B-I\\_{{OCR}}", "MQ3-VL-4B-I\\_{{Conv.}}"])]
clean_df.Model = clean_df.Model.apply(lambda x: f"\\rotatebox{{90}}{{{x}}}")

clean_df = clean_df.set_index(["Model", "Tasks"])
clean_df = clean_df.round(3)

clean_df

Valid. (\%)  EM (\%)  CSM (\%)  \
Model                       Tasks                                         
\rotatebox{90}{Q3-4B-I}     Rep. Trans.       14.980    0.000     0.000   
                            IUPAC Und.        27.536    0.000     0.000   
                            CML Und.          11.842    0.000     0.000   
                            Intra Norm.       49.123    0.000    30.702   
                            Cross Norm.       15.176    0.000     0.000   
                            Overall           20.229    0.000     4.000   
\rotatebox{90}{Q3-VL-4B-I}  Rep. Trans.        5.668    0.405     0.405   
                            IUPAC Und.        13.043    0.000     0.000   
                            CML Und.          17.105    0.000     0.000   
                            Intra Norm.       70.175    0.000    53.509   
                            Cross Norm.        8.672    0.000     0.000   
                            Overall           16.914    0.114     7.086   
\rotatebox{90}{MQ3-4B-I}    Rep. Trans.       88.664   68.421    68.421   
                            IUPAC Und.        73.913   37.681    37.681   
                            CML Und.          77.632   56.579    56.579   
                            Intra Norm.       98.246   80.702    80.702   
                            Cross Norm.       95.664   88.618    88.618   
                            Overall           90.743   75.086    75.086   
\rotatebox{90}{MQ3-VL-4B-I} Rep. Trans.       91.093   67.611    67.611   
                            IUPAC Und.        76.812   36.232    36.232   
                            CML Und.          85.526   56.579    56.579   
                            Intra Norm.       98.246   77.193    77.193   
                            Cross Norm.       92.954   85.908    85.908   
                            Overall           91.200   73.143    73.143   

                                         IKM (\%)  T. Sim  
Model                       Tasks                          
\rotatebox{90}{Q3-4B-I}     Rep. Trans.     0.000   0.007  
                            IUPAC Und.      0.000   0.025  
                            CML Und.        0.000   0.004  
                            Intra Norm.    30.702   0.385  
                            Cross Norm.     0.000   0.014  
                            Overall         4.000   0.060  
\rotatebox{90}{Q3-VL-4B-I}  Rep. Trans.     0.405   0.011  
                            IUPAC Und.      0.000   0.020  
                            CML Und.        0.000   0.010  
                            Intra Norm.    53.509   0.617  
                            Cross Norm.     0.000   0.012  
                            Overall         7.086   0.091  
\rotatebox{90}{MQ3-4B-I}    Rep. Trans.    68.421   0.758  
                            IUPAC Und.     37.681   0.495  
                            CML Und.       56.579   0.645  
                            Intra Norm.    80.702   0.875  
                            Cross Norm.    88.618   0.915  
                            Overall        75.086   0.809  
\rotatebox{90}{MQ3-VL-4B-I} Rep. Trans.    67.611   0.754  
                            IUPAC Und.     36.232   0.502  
                            CML Und.       56.579   0.714  
                            Intra Norm.    77.193   0.854  
                            Cross Norm.    85.908   0.891  
                            Overall        73.143   0.802

In [25]:
print(clean_df.to_latex(
    float_format="%.3f",
    index=True,
))

\begin{tabular}{llrrrrr}
\toprule
 &  & Valid. (\%) & EM (\%) & CSM (\%) & IKM (\%) & T. Sim \\
Model & Tasks &  &  &  &  &  \\
\midrule
\multirow[t]{6}{*}{\rotatebox{90}{Q3-4B-I}} & Rep. Trans. & 14.980 & 0.000 & 0.000 & 0.000 & 0.007 \\
 & IUPAC Und. & 27.536 & 0.000 & 0.000 & 0.000 & 0.025 \\
 & CML Und. & 11.842 & 0.000 & 0.000 & 0.000 & 0.004 \\
 & Intra Norm. & 49.123 & 0.000 & 30.702 & 30.702 & 0.385 \\
 & Cross Norm. & 15.176 & 0.000 & 0.000 & 0.000 & 0.014 \\
 & Overall & 20.229 & 0.000 & 4.000 & 4.000 & 0.060 \\
\cline{1-7}
\multirow[t]{6}{*}{\rotatebox{90}{Q3-VL-4B-I}} & Rep. Trans. & 5.668 & 0.405 & 0.405 & 0.405 & 0.011 \\
 & IUPAC Und. & 13.043 & 0.000 & 0.000 & 0.000 & 0.020 \\
 & CML Und. & 17.105 & 0.000 & 0.000 & 0.000 & 0.010 \\
 & Intra Norm. & 70.175 & 0.000 & 53.509 & 53.509 & 0.617 \\
 & Cross Norm. & 8.672 & 0.000 & 0.000 & 0.000 & 0.012 \\
 & Overall & 16.914 & 0.114 & 7.086 & 7.086 & 0.091 \\
\cline{1-7}
\multirow[t]{6}{*}{\rotatebox{90}{MQ3-4B-I}} & Rep. Tran

### Table3: Base model v.s. Trained Model: Input Rep v.s. Output Rep

In [26]:
tmp_df = conversion_df.groupby(["Model Version", "Model", "Input Rep. Type", "Output Rep. Type"]).apply(aggregate_func).reset_index()
tmp_df = tmp_df.rename(columns=agg_col_names)
tmp_df

,Model Version,Model,Input Rep. Type,Output Rep. Type,N,GT Valid. (\%),Valid. (\%),EM (\%),CSM (\%),IKM (\%),T. Sim,T. Sim (std)
0,Base,Q3-4B-I,SMILES,SELFIES,18.0,100.0,38.888889,0.000000,0.000000,0.000000,0.000000,0.000000
1,Base,Q3-4B-I,SMILES,DeepSMILES,17.0,100.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,Base,Q3-4B-I,SMILES,InChI,19.0,100.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,Base,Q3-4B-I,SELFIES,SMILES,16.0,100.0,18.750000,0.000000,0.000000,0.000000,0.020057,0.044538
4,Base,Q3-4B-I,SELFIES,DeepSMILES,25.0,100.0,8.000000,0.000000,0.000000,0.000000,0.004035,0.016888
...,...,...,...,...,...,...,...,...,...,...,...,...
187,Trained,MQ3-VL-4B-I,IUPAC,InChI,20.0,100.0,50.000000,30.000000,30.000000,30.000000,0.416321,0.458173
188,Trained,MQ3-VL-4B-I,CML,SMILES,17.0,100.0,94.117647,52.941176,52.941176,52.941176,0.753076,0.344741
189,Trained,MQ3-VL-4B-I,CML,SELFIES,17.0,100.0,100.000000,52.941176,52.941176,52.941176,0.779866,0.363974
190,Trained,MQ3-VL-4B-I,CML,DeepSMILES,24.0,100.0,83.333333,62.500000,62.500000,62.500000,0.731345,0.401193


In [29]:
clean_df = tmp_df.copy()
clean_df = clean_df[~clean_df.Model.isin(["MQ3-VL-4B-I\\_{{OCR}}", "MQ3-VL-4B-I\\_{{Conv.}}"])]
clean_df = clean_df.drop(columns=["N", "GT Valid. (\\%)", "EM (\\%)", "CSM (\\%)", "T. Sim (std)"])
clean_df = clean_df.pivot_table(index=["Model Version", "Input Rep. Type"], columns=["Output Rep. Type"], values=["Valid. (\\%)", "IKM (\\%)", "T. Sim"])
clean_df.columns = clean_df.columns.swaplevel(0, 1)
clean_df = clean_df.sort_index(axis=1, level=0)

clean_df

Output Rep. Type                  SMILES                          SELFIES  \
                                IKM (\%)    T. Sim Valid. (\%)   IKM (\%)   
Model Version Input Rep. Type                                               
Base          SMILES                 NaN       NaN         NaN   0.000000   
              SELFIES           0.000000  0.015894   18.750000        NaN   
              DeepSMILES        2.272727  0.059937   22.727273   0.000000   
              InChI             0.000000  0.026944   26.190476   0.000000   
              R. SMILES        30.882353  0.539079   77.941176   0.000000   
              R. SELFIES        0.000000  0.025542   22.222222  73.863636   
              R. DeepSMILES     0.000000  0.079433   37.500000   0.000000   
              IUPAC             0.000000  0.118902   57.692308   0.000000   
              CML               0.000000  0.024370   32.352941   0.000000   
Trained       SMILES                 NaN       NaN         NaN  86.111111   
              SELFIES          84.375000  0.899819   96.875000        NaN   
              DeepSMILES       86.363636  0.946544  100.000000  81.578947   
              InChI            19.047619  0.408264   80.952381   7.142857   
              R. SMILES        95.588235  0.995541  100.000000  80.882353   
              R. SELFIES       83.333333  0.879114   96.666667  60.227273   
              R. DeepSMILES    94.318182  0.966470   98.863636  80.000000   
              IUPAC            46.153846  0.634591   88.461538  27.272727   
              CML              55.882353  0.716144   88.235294  52.941176   

Output Rep. Type                                    DeepSMILES            \
                                 T. Sim Valid. (\%)   IKM (\%)    T. Sim   
Model Version Input Rep. Type                                              
Base          SMILES           0.000000   25.000000   0.000000  0.000000   
              SELFIES               NaN         NaN   0.000000  0.002018   
              DeepSMILES       0.000000   15.789474        NaN       NaN   
              InChI            0.000821   16.666667   0.000000  0.000000   
              R. SMILES        0.000000   16.176471   0.000000  0.000000   
              R. SELFIES       0.763499   79.545455   0.000000  0.000000   
              R. DeepSMILES    0.000000   30.000000  13.888889  0.144367   
              IUPAC            0.000000   29.545455   0.000000  0.000000   
              CML              0.005045   29.411765   0.000000  0.001225   
Trained       SMILES           0.928563  100.000000  79.411765  0.808088   
              SELFIES               NaN         NaN  90.000000  0.935359   
              DeepSMILES       0.869052  100.000000        NaN       NaN   
              InChI            0.265673   90.476190  16.666667  0.321428   
              R. SMILES        0.842373   94.117647  88.636364  0.962744   
              R. SELFIES       0.696463   95.454545  85.483871  0.901496   
              R. DeepSMILES    0.851192   95.714286  86.111111  0.947451   
              IUPAC            0.450673  100.000000  50.000000  0.567448   
              CML              0.720376  100.000000  66.666667  0.747834   

Output Rep. Type                                InChI                        
                              Valid. (\%)    IKM (\%)    T. Sim Valid. (\%)  
Model Version Input Rep. Type                                                
Base          SMILES             0.000000    0.000000  0.000000    0.000000  
              SELFIES            4.000000    0.000000  0.000000    0.000000  
              DeepSMILES              NaN    0.000000  0.000000    0.000000  
              InChI              0.000000         NaN       NaN         NaN  
              R. SMILES          0.000000    0.000000  0.000000    0.000000  
              R. SELFIES         1.612903    0.000000  0.000000    0.000000  
              R. DeepSMILES     18.055556    0.000000  0.001322    2.040816  
     

In [30]:
print(clean_df.to_latex(
    float_format="%.3f",
    index=True,
    na_rep="-",
))

\begin{tabular}{llrrrrrrrrrrrr}
\toprule
 & Output Rep. Type & \multicolumn{3}{r}{SMILES} & \multicolumn{3}{r}{SELFIES} & \multicolumn{3}{r}{DeepSMILES} & \multicolumn{3}{r}{InChI} \\
 &  & IKM (\%) & T. Sim & Valid. (\%) & IKM (\%) & T. Sim & Valid. (\%) & IKM (\%) & T. Sim & Valid. (\%) & IKM (\%) & T. Sim & Valid. (\%) \\
Model Version & Input Rep. Type &  &  &  &  &  &  &  &  &  &  &  &  \\
\midrule
\multirow[t]{9}{*}{Base} & SMILES & - & - & - & 0.000 & 0.000 & 25.000 & 0.000 & 0.000 & 0.000 & 0.000 & 0.000 & 0.000 \\
 & SELFIES & 0.000 & 0.016 & 18.750 & - & - & - & 0.000 & 0.002 & 4.000 & 0.000 & 0.000 & 0.000 \\
 & DeepSMILES & 2.273 & 0.060 & 22.727 & 0.000 & 0.000 & 15.789 & - & - & - & 0.000 & 0.000 & 0.000 \\
 & InChI & 0.000 & 0.027 & 26.190 & 0.000 & 0.001 & 16.667 & 0.000 & 0.000 & 0.000 & - & - & - \\
 & R. SMILES & 30.882 & 0.539 & 77.941 & 0.000 & 0.000 & 16.176 & 0.000 & 0.000 & 0.000 & 0.000 & 0.000 & 0.000 \\
 & R. SELFIES & 0.000 & 0.026 & 22.222 & 73.864 & 0.763 

### Table2: VLM on OCR

In [31]:
tmp_df = ocr_df.groupby(["Model", "Tasks"]).apply(aggregate_func).reset_index()
tmp_df = tmp_df.rename(columns=agg_col_names)

tmp_df = tmp_df.round(3)
tmp_df

,Model,Tasks,N,GT Valid. (\%),Valid. (\%),EM (\%),CSM (\%),IKM (\%),T. Sim,T. Sim (std)
0,Q3-VL-4B-I,OCR,80.0,100.0,23.75,0.00,1.25,1.25,0.075,0.198
1,MQ3-VL-4B-I\_{{OCR}},OCR,80.0,100.0,43.75,3.75,3.75,3.75,0.179,0.298
2,MQ3-VL-4B-I\_{{Conv.}},OCR,80.0,100.0,63.75,20.00,20.00,20.00,0.338,0.394
3,MQ3-VL-4B-I,OCR,80.0,100.0,68.75,26.25,26.25,26.25,0.411,0.416


In [32]:
clean_df = tmp_df.copy()
clean_df = clean_df[~clean_df.Model.isin(["MQ3-VL-4B-I\\_{{OCR}}", "MQ3-VL-4B-I\\_{{Conv.}}"])]
clean_df = clean_df.drop(columns=["N", "GT Valid. (\\%)", "T. Sim (std)"])
# clean_df = clean_df.pivot_table(index=["Model Version", "Input Rep. Type"], columns=["Output Rep. Type"], values=["Valid.", "IKM", "T. Sim"])
# clean_df.columns = clean_df.columns.swaplevel(0, 1)
# clean_df = clean_df.sort_index(axis=1, level=0)

clean_df

,Model,Tasks,Valid. (\%),EM (\%),CSM (\%),IKM (\%),T. Sim
0,Q3-VL-4B-I,OCR,23.75,0.00,1.25,1.25,0.075
3,MQ3-VL-4B-I,OCR,68.75,26.25,26.25,26.25,0.411


In [34]:
print(clean_df.to_latex(
    float_format="%.2f",
    index=False,
))

\begin{tabular}{llrrrrr}
\toprule
Model & Tasks & Valid. (\%) & EM (\%) & CSM (\%) & IKM (\%) & T. Sim \\
\midrule
Q3-VL-4B-I & OCR & 23.75 & 0.00 & 1.25 & 1.25 & 0.07 \\
MQ3-VL-4B-I & OCR & 68.75 & 26.25 & 26.25 & 26.25 & 0.41 \\
\bottomrule
\end{tabular}



### Table4: LLM v.s. VLM

In [35]:
combined_df = pd.concat([conversion_df, ocr_df], ignore_index=True)
combined_df["Test Subset"] = "In-Domain"

extra_combined_df = pd.concat([extra_conversion_df, extra_ocr_df], ignore_index=True)
extra_combined_df["Test Subset"] = "Out-of-Domain"

big_combined_df = pd.concat([combined_df, extra_combined_df], ignore_index=True)

In [36]:
test_subset_order = ["In-Domain", "Out-of-Domain"]
big_combined_df["Test Subset"] = pd.Categorical(big_combined_df["Test Subset"], categories=test_subset_order, ordered=True)

tmp_col_names = {
    "model_modality": "Model Type",
    "task_name": "Task Type",
}
model_modality_map = {
    "llm": "LLM",
    "vlm": "MLLM",
}
model_modality_order = ["LLM", "MLLM"]
task_name_map = {
    "mol_rep_conversion": "Conv.",
    "mol_rep_ocr": "OCR",
}
task_name_order = ["Conv.", "OCR"]

big_combined_df = big_combined_df.rename(columns=tmp_col_names)

big_combined_df["Model Type"] = big_combined_df["Model Type"].replace(model_modality_map)
big_combined_df["Model Type"] = pd.Categorical(big_combined_df["Model Type"], categories=model_modality_order, ordered=True)
big_combined_df["Task Type"] = big_combined_df["Task Type"].replace(task_name_map)
big_combined_df["Task Type"] = pd.Categorical(big_combined_df["Task Type"], categories=task_name_order, ordered=True)

In [37]:
tmp_df = big_combined_df.groupby(["Test Subset", "Model Type", "Task Type", "Model"]).apply(aggregate_func).reset_index()
tmp_df = tmp_df.rename(columns=agg_col_names)

tmp_df = tmp_df.round(2)
tmp_df

,Test Subset,Model Type,Task Type,Model,N,GT Valid. (\%),Valid. (\%),EM (\%),CSM (\%),IKM (\%),T. Sim,T. Sim (std)
0,In-Domain,LLM,Conv.,Q3-4B-I,875.0,100.0,20.23,0.00,4.00,4.00,0.06,0.21
1,In-Domain,LLM,Conv.,MQ3-4B-I,875.0,100.0,90.74,75.09,75.09,75.09,0.81,0.36
2,In-Domain,MLLM,Conv.,Q3-VL-4B-I,875.0,100.0,16.91,0.11,7.09,7.09,0.09,0.27
3,In-Domain,MLLM,Conv.,MQ3-VL-4B-I\_{{OCR}},875.0,100.0,25.49,0.00,5.03,5.03,0.09,0.24
4,In-Domain,MLLM,Conv.,MQ3-VL-4B-I\_{{Conv.}},875.0,100.0,89.37,70.86,70.86,70.86,0.78,0.38
5,In-Domain,MLLM,Conv.,MQ3-VL-4B-I,875.0,100.0,91.20,73.14,73.14,73.14,0.80,0.36
6,In-Domain,MLLM,OCR,Q3-VL-4B-I,80.0,100.0,23.75,0.00,1.25,1.25,0.08,0.20
7,In-Domain,MLLM,OCR,MQ3-VL-4B-I\_{{OCR}},80.0,100.0,43.75,3.75,3.75,3.75,0.18,0.30
8,In-Domain,MLLM,OCR,MQ3-VL-4B-I\_{{Conv.}},80.0,100.0,63.75,20.00,20.00,20.00,0.34,0.39
9,In-Domain,MLLM,OCR,MQ3-VL-4B-I,80.0,100.0,68.75,26.25,26.25,26.25,0.41,0.42


In [41]:
clean_df = tmp_df.copy()
# clean_df = clean_df[clean_df["Task Type"] == "Conv."]
clean_df = clean_df.drop(columns=["N", "GT Valid. (\\%)", "EM (\\%)", "CSM (\\%)"])
# clean_df = clean_df.pivot_table(index=["Model Version", "Input Rep. Type"], columns=["Output Rep. Type"], values=["Valid.", "IKM", "T. Sim"])
# clean_df.columns = clean_df.columns.swaplevel(0, 1)
# clean_df = clean_df.sort_index(axis=1, level=0)
clean_df["Test Subset"] = clean_df["Test Subset"].apply(lambda x: f"\\rotatebox{{90}}{{{x}}}")
clean_df["Model Type"] = clean_df["Model Type"].apply(lambda x: f"\\rotatebox{{90}}{{{x}}}")
clean_df["Task Type"] = clean_df["Task Type"].apply(lambda x: f"\\rotatebox{{90}}{{{x}}}")

clean_df = clean_df.set_index(["Test Subset", "Model Type", "Task Type", "Model"])

clean_df

Valid. (\%)  \
Test Subset                   Model Type           Task Type             Model                                 
\rotatebox{90}{In-Domain}     \rotatebox{90}{LLM}  \rotatebox{90}{Conv.} Q3-4B-I                       20.23   
                                                                         MQ3-4B-I                      90.74   
                              \rotatebox{90}{MLLM} \rotatebox{90}{Conv.} Q3-VL-4B-I                    16.91   
                                                                         MQ3-VL-4B-I\_{{OCR}}          25.49   
                                                                         MQ3-VL-4B-I\_{{Conv.}}        89.37   
                                                                         MQ3-VL-4B-I                   91.20   
                                                   \rotatebox{90}{OCR}   Q3-VL-4B-I                    23.75   
                                                                         MQ3-VL-4B-I\_{{OCR}}          43.75   
                                                                         MQ3-VL-4B-I\_{{Conv.}}        63.75   
                                                                         MQ3-VL-4B-I                   68.75   
\rotatebox{90}{Out-of-Domain} \rotatebox{90}{LLM}  \rotatebox{90}{Conv.} Q3-4B-I                       20.34   
                                                                         MQ3-4B-I                      58.18   
                              \rotatebox{90}{MLLM} \rotatebox{90}{Conv.} Q3-VL-4B-I                    17.16   
                                                                         MQ3-VL-4B-I\_{{OCR}}          29.77   
                                                                         MQ3-VL-4B-I\_{{Conv.}}        62.39   
                                                                         MQ3-VL-4B-I                   59.43   
                                                   \rotatebox{90}{OCR}   Q3-VL-4B-I                    18.75   
                                                                         MQ3-VL-4B-I\_{{OCR}}          41.25   
                                                                         MQ3-VL-4B-I\_{{Conv.}}        58.75   
                                                                         MQ3-VL-4B-I                   55.00   

                                                                                                 IKM (\%)  \
Test Subset                   Model Type           Task Type             Model                              
\rotatebox{90}{In-Domain}     \rotatebox{90}{LLM}  \rotatebox{90}{Conv.} Q3-4B-I                     4.00   
                                                                         MQ3-4B-I                   75.09   
                              \rotatebox{90}{MLLM} \rotatebox{90}{Conv.} Q3-VL-4B-I                  7.09   
                                                                         MQ3-VL-4B-I\_{{OCR}}        5.03   
                                                                         MQ3-VL-4B-I\_{{Conv.}}     70.86   
                                                                         MQ3-VL-4B-I                73.14   
                                                   \rotatebox{90}{OCR}   Q3-VL-4B-I                  1.25   
                                                                         MQ3-VL-4B-I\_{{OCR}}        3.75   
                                                                         MQ3-VL-4B-I\_{{Conv.}}     20.00   
                                                                         MQ3-VL-4B-I                26.25   
\rotatebox{90}{Out-of-Domain} \rotatebox{90}{LLM}  \rotatebox{90}{Conv.} Q3-4B-I                     3.86   
                                                                         MQ3-4B-I                    3.07   
                              \rotatebox{90}{MLLM} \rotatebox{90}{Conv.} Q3-VL-4B-I                  6.48 

In [42]:
print(clean_df.to_latex(
    float_format="%.2f",
    index=True,
))

\begin{tabular}{llllrrrr}
\toprule
 &  &  &  & Valid. (\%) & IKM (\%) & T. Sim & T. Sim (std) \\
Test Subset & Model Type & Task Type & Model &  &  &  &  \\
\midrule
\multirow[t]{10}{*}{\rotatebox{90}{In-Domain}} & \multirow[t]{2}{*}{\rotatebox{90}{LLM}} & \multirow[t]{2}{*}{\rotatebox{90}{Conv.}} & Q3-4B-I & 20.23 & 4.00 & 0.06 & 0.21 \\
 &  &  & MQ3-4B-I & 90.74 & 75.09 & 0.81 & 0.36 \\
\cline{2-8} \cline{3-8}
 & \multirow[t]{8}{*}{\rotatebox{90}{MLLM}} & \multirow[t]{4}{*}{\rotatebox{90}{Conv.}} & Q3-VL-4B-I & 16.91 & 7.09 & 0.09 & 0.27 \\
 &  &  & MQ3-VL-4B-I\_{{OCR}} & 25.49 & 5.03 & 0.09 & 0.24 \\
 &  &  & MQ3-VL-4B-I\_{{Conv.}} & 89.37 & 70.86 & 0.78 & 0.38 \\
 &  &  & MQ3-VL-4B-I & 91.20 & 73.14 & 0.80 & 0.36 \\
\cline{3-8}
 &  & \multirow[t]{4}{*}{\rotatebox{90}{OCR}} & Q3-VL-4B-I & 23.75 & 1.25 & 0.08 & 0.20 \\
 &  &  & MQ3-VL-4B-I\_{{OCR}} & 43.75 & 3.75 & 0.18 & 0.30 \\
 &  &  & MQ3-VL-4B-I\_{{Conv.}} & 63.75 & 20.00 & 0.34 & 0.39 \\
 &  &  & MQ3-VL-4B-I & 68.75 & 26.25 & 0

## Rollout

## MolRepBench

In [ ]:
from pathlib import Path
import pandas as pd

tmp_dir = Path("../molrepbench/results_small")
cross_dir = tmp_dir / "cross"
metrics_dir = tmp_dir / "metrics"

In [ ]:
tmp_file = cross_dir / "comprehension_vs_generation.csv"
tmp_df = pd.read_csv(tmp_file)

tmp_df.groupby(["model"])[["b1_score", "b2_score", "b3_score", "b4_score", "b5_score", "b6_score", "mean_comprehension", "mean_generation"]].mean()

,b1_score,b2_score,b3_score,b4_score,b5_score,b6_score,mean_comprehension,mean_generation
model,,,,,,,,
molqwen3-4b-instruct-sft,0.252222,0.269596,0.620268,0.575499,0.068281,0.313248,0.357173,0.313248
molqwen3-vl-4b-instruct-sft,0.227778,0.280452,0.574110,0.598291,0.011975,0.313404,0.338521,0.313404
qwen3-4b-instruct-2507,0.251667,0.247047,0.689895,0.568376,0.001602,0.240649,0.351718,0.240649
qwen3-vl-4b-instruct,0.297778,0.285862,0.405191,0.555556,0.003120,0.258793,0.309501,0.258793


In [ ]:
tmp_file = metrics_dir / "benchmark_9_metrics.csv"
tmp_df = pd.read_csv(tmp_file)

tmp_df[tmp_df["pair_type"] == "overall"].groupby(["model"])[["accuracy", "precision", "recall", "f1"]].mean()


,accuracy,precision,recall,f1
model,,,,
molqwen3-4b-instruct-sft,0.828444,0.886131,0.745778,0.802304
molqwen3-vl-4b-instruct-sft,0.758222,0.967155,0.534222,0.676924
qwen3-4b-instruct-2507,0.631111,1.000000,0.262222,0.357240
qwen3-vl-4b-instruct,0.597778,1.000000,0.195556,0.268804


In [ ]:
tmp_file = metrics_dir / "benchmark_10_metrics.csv"
tmp_df = pd.read_csv(tmp_file)

tmp_df[tmp_df["pair_type"] == "overall"].groupby(["model"])[["accuracy", "precision", "recall", "f1"]].mean()

,accuracy,precision,recall,f1
model,,,,
molqwen3-4b-instruct-sft,0.646551,0.709760,0.440212,0.523372
molqwen3-vl-4b-instruct-sft,0.819821,0.982127,0.651852,0.775642
qwen3-4b-instruct-2507,0.831598,0.992390,0.667725,0.763244
qwen3-vl-4b-instruct,0.693473,1.000000,0.387302,0.519876
